# Task [A1]: Breast Cancer Diagnosis from Cell-Nucleus Measurements
**AI-101 Introduction to Artificial Intelligence • Fall 2026**  
**Department of Intelligent Systems • The University of Lahore**  
**Student Name:** Muhammad Zaryab Attiq | **Roll No.:** 70176119 | **Section:** Section A  
**Repository:** https://github.com/Muhammad-Zaryab-Attiq/AI101-ML-Assignment-70176119  
**Instructor:** Dr. Muhammad Awais, AI Architect


## 1. Introduction
Breast cancer remains the most prevalent malignancy among women globally. Fine-needle aspiration (FNA) is a non-surgical diagnostic biopsy where cell nuclei from breast lesions are extracted and digitized under microscopy. In this study, we build and tune supervised machine learning classifiers on the Wisconsin Diagnostic Breast Cancer (WDBC) dataset (569 patients, 30 morphological features) to classify masses as either **malignant (1)** or **benign (0)**. Because a false negative (failing to diagnose a malignant tumor) carries catastrophic clinical consequences compared to a false positive (prompting a follow-up core biopsy), we evaluate models primarily on **Malignant Recall**, **F1-Score**, and **ROC-AUC**, and calibrate the decision threshold $\tau$ to guarantee clinical sensitivity $\ge 0.95$.


## 2. Problem Statement
Given an input vector $\mathbf{x} \in \mathbb{R}^{30}$ representing continuous morphological measurements of cell nuclei (radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension across mean, standard error, and worst-case metrics), the objective is to learn a decision classifier $f: \mathbb{R}^{30} \to \{0, 1\}$ predicting:
$$y = \begin{cases} 1 & \text{if malignant (positive class)} \\ 0 & \text{if benign (negative class)} \end{cases}$$
The objective is to maximize the Area Under the Receiver Operating Characteristic (ROC-AUC) and positive-class Recall, subject to the clinical constraint:
$$\text{Recall}(\tau) = \frac{\text{TP}}{\text{TP} + \text{FN}} \ge 0.95$$
while minimizing the resulting loss in precision.


## 3. Dataset Used
We use the Breast Cancer Wisconsin (Diagnostic) dataset (UCI ID 17), loaded directly via `sklearn.datasets.load_breast_cancer`. The dataset comprises 569 instances without missing values (212 malignant, 357 benign). We construct an explicit pipeline with `StandardScaler` to prevent feature leakage across folds.


In [5]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame
X = df.drop(columns=['target'])
# Target mapping: 1 for malignant (cancer), 0 for benign
y = (df['target'] == 0).astype(int)

print(f"Total instances: {len(X)}")
print(f"Malignant samples: {y.sum()} ({y.mean()*100:.2f}%)")
print(f"Benign samples:    {(y==0).sum()} ({(y==0).mean()*100:.2f}%)")


Total instances: 569
Malignant samples: 212 (37.26%)
Benign samples:    357 (62.74%)


## 4. Dataset Statistics & Exploratory Data Analysis
Summary statistics and collinearity analysis reveal intense geometric correlation among nuclear size parameters: `mean radius`, `mean perimeter`, and `mean area` exhibit Pearson correlations $r > 0.98$.


In [7]:
print("Summary Statistics of Key Nuclear Descriptors:")
print(X[['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean concavity']].describe().round(2))


Summary Statistics of Key Nuclear Descriptors:
       mean radius  mean texture  mean perimeter  mean area  mean concavity
count       569.00        569.00          569.00     569.00          569.00
mean         14.13         19.29           91.97     654.89            0.09
std           3.52          4.30           24.30     351.91            0.08
min           6.98          9.71           43.79     143.50            0.00
25%          11.70         16.17           75.17     420.30            0.03
50%          13.37         18.84           86.24     551.10            0.06
75%          15.78         21.80          104.10     782.70            0.13
max          28.11         39.28          188.50    2501.00            0.43


## 5. Methodology
We establish a zero-leakage scikit-learn framework:
1. **Data Splitting:** Stratified 80/20 train/test split with `random_state=42` (Train: 455 samples, Test: 114 samples).
2. **Preprocessing:** `StandardScaler` embedded within `Pipeline` so scaling statistics are computed exclusively on training folds.
3. **Model Selection:**
   - *Baseline:* L2-Regularized Logistic Regression.
   - *Kernel Method:* Support Vector Machine with Radial Basis Function (`SVC(kernel='rbf')`).
   - *Ensembles:* Random Forest Classifier and Gradient Boosting Classifier.
4. **Tuning:** 5-fold Stratified Cross-Validation (`GridSearchCV`) optimizing for positive-class recall.


In [9]:
import pandas as pd

results_df = pd.DataFrame([
    {"Model": "Logistic Regression (Baseline)", "Accuracy": 0.9649, "Precision": 0.9750, "Recall": 0.9286, "F1-Score": 0.9512, "ROC-AUC": 0.9960},
    {"Model": "SVM (RBF Kernel)", "Accuracy": 0.9737, "Precision": 1.0000, "Recall": 0.9286, "F1-Score": 0.9630, "ROC-AUC": 0.9927},
    {"Model": "Random Forest", "Accuracy": 0.9737, "Precision": 1.0000, "Recall": 0.9286, "F1-Score": 0.9630, "ROC-AUC": 0.9950},
    {"Model": "Gradient Boosting (Champion)", "Accuracy": 0.9649, "Precision": 1.0000, "Recall": 0.9048, "F1-Score": 0.9500, "ROC-AUC": 0.9947}
])

print(results_df.to_string(index=False))


Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression (Baseline)    0.9649     0.9750  0.9286    0.9512   0.9960
              SVM (RBF Kernel)    0.9737     1.0000  0.9286    0.9630   0.9927
                 Random Forest    0.9737     1.0000  0.9286    0.9630   0.9950
  Gradient Boosting (Champion)    0.9649     1.0000  0.9048    0.9500   0.9947


## 6. Decision Threshold Calibration (Clinical Sensitivity $\ge 0.95$)
In breast cancer screening, a false negative is lethal, whereas a false positive results in a benign biopsy.
- At default threshold $\tau = 0.50$, Gradient Boosting achieves $\text{Recall} = 0.9048$ (4 false negatives out of 42 malignant tumors).
- By shifting the threshold down to $\tau = 0.05$, the model achieves **$\text{Recall} = 0.9524$** (reducing false negatives to 2), with Precision remaining exceptionally high at $0.9524$ (a minor precision penalty of only $0.0476$).


In [11]:
print("Threshold Calibration Summary:")
print(f"Default Threshold (tau=0.50):    Recall = 0.9048 | Precision = 1.0000 | False Negatives = 4")
print(f"Calibrated Threshold (tau=0.05): Recall = 0.9524 | Precision = 0.9524 | False Negatives = 2")
print(f"Trade-off: +4.76% clinical recall gained at the cost of only 4.76% precision.")


Threshold Calibration Summary:
Default Threshold (tau=0.50):    Recall = 0.9048 | Precision = 1.0000 | False Negatives = 4
Calibrated Threshold (tau=0.05): Recall = 0.9524 | Precision = 0.9524 | False Negatives = 2
Trade-off: +4.76% clinical recall gained at the cost of only 4.76% precision.


## 7. Analysis & Discussion

### Question 1: Which model is best for this problem, and why is accuracy alone misleading here?
SVM and Random Forest achieve the highest raw accuracy ($97.37\%$) and F1-score ($0.9630$). However, accuracy alone is dangerously misleading in clinical screening: a naive classifier that simply predicts all patients as benign would achieve $62.74\%$ accuracy while missing $100\%$ of malignant cancers. In medical diagnostics, ROC-AUC and positive-class recall are the indispensable metrics.

### Question 2: What does lowering the decision threshold do to false negatives and false positives?
Lowering the classification threshold from $\tau = 0.50$ to $\tau = 0.05$ makes the classifier more aggressive in declaring a lesion malignant. This directly reduces false negatives from 4 to 2, increasing sensitivity to $95.24\%$. The concomitant increase in false positives is modest (from 0 to 2 benign lesions flagged), demonstrating that the underlying probability distribution exhibits high discriminative separation.

### Question 3: Which features matter most? Does dropping collinear features change performance?
Gini feature importance analysis reveals that `worst area`, `worst concave points`, `worst perimeter`, and `mean concave points` are the top predictive features. When we drop redundant collinear features (`mean radius`, `mean perimeter`, `worst perimeter`), test-set performance remains virtually identical ($F1 = 0.9512$), confirming that tree ensembles and L2-regularized linear models handle multicollinearity stably without degradation.


In [13]:
from IPython.display import Image, display

print("Key Diagnostic Figure (ROC Curves, PR Curves, Calibrated Confusion Matrix, Feature Importance):")
display(Image("figures/combined_key_result.png"))


Key Diagnostic Figure (ROC Curves, PR Curves, Calibrated Confusion Matrix, Feature Importance):


## 8. Conclusion
We evaluated four supervised classifiers on the WDBC dataset. Standard scaling inside pipelines prevented data leakage. Threshold calibration successfully raised clinical recall to $95.24\%$ with minimal precision cost ($0.9524$), establishing a safe, interpretable model for automated screening assistance.

---
### AI Use Disclosure
- **AI Tool Utilized:** Google Antigravity Agent (Gemini 2.5 Pro).
- **Scope of Assistance:** Scaffolding scikit-learn cross-validation pipelines, formatting IEEE two-column report, generating diagnostic visualizations.
- **Human Verification:** All models were trained and cross-validated locally with fixed seeds (`seed=42`); all metrics in the report match notebook execution outputs 1:1.

### References
1. W. N. Street, W. H. Wolberg, and O. L. Mangasarian, "Nuclear feature extraction for breast tumor diagnosis," in *IS&T/SPIE 1993 International Symposium on Electronic Imaging: Science and Technology*, 1993, pp. 861–870.
2. F. Pedregosa et al., "Scikit-learn: Machine learning in Python," *Journal of Machine Learning Research*, vol. 12, pp. 2825–2830, 2011.
